# 自然语言处理 (NLP) 词表征算法企业笔试手撕通关宝典
> **面向对象**：互联网大厂/AI 独角兽企业 NLP 算法岗、大模型前置算法手撕面试  
> **核心涵盖**：共现矩阵、PPMI(向量化加速)、SVD降维、高频词下采样、负采样轮盘查表法、Embedding梯度原子累加、CBOW/Skip-Gram核心实现、GloVe加权回归、自注意力动态词表征  
> **设计准则**：纯 NumPy 极简高能实现，剔除冗余框架依赖，突出核心数据流、求导闭式解与高频笔试陷阱。

---
### 核心模块速览
1. **模块一**：滑动窗口共现矩阵构建 (Co-occurrence Matrix)
2. **模块二**：正点互信息 (PPMI) 纯向量化广播手撕
3. **模块三**：截断 SVD 降维稠密向量与余弦相似度
4. **模块四**：Mikolov 高频词二次物理下采样 (Subsampling)
5. **模块五**：工业级负采样器 (0.75 次幂平滑 + 100万槽位轮盘表)
6. **模块六**：Embedding 层与反向传播【梯度覆盖毁灭陷阱】防范
7. **模块七**：CBOW + 负采样端到端手撕 (前向池化 + 二分类损失 + 梯度反传)
8. **模块八**：Skip-Gram + 负采样端到端手撕 (点对点单挑解耦)
9. **模块九**：GloVe 全局对数共现加权最小二乘回归手撕
10. **模块十**：现代动态上下文表征——缩放点积自注意力 (Self-Attention)


---
## 模块一：滑动窗口与共现矩阵构建 (Co-occurrence Matrix)

### 【笔试考点与陷阱】
1. **双向滑动窗口边界**：左侧不能越界 `< 0`，右侧不能越界 `>= len(corpus)`；
2. **词表映射**：构建双向映射字典 `word_to_id` 与 `id_to_word`；
3. **对称性**：未加权时，$C_{ij} = C_{ji}$。


In [ ]:
import numpy as np
import collections

def build_vocab(words):
    """构建词表映射字典"""
    word_to_id, id_to_word = {}, {}
    for word in words:
        if word not in word_to_id:
            idx = len(word_to_id)
            word_to_id[word] = idx
            id_to_word[idx] = word
    return word_to_id, id_to_word

def create_co_matrix(corpus, vocab_size, window_size=1):
    """
    根据滑动窗口统计共现频次矩阵
    参数:
        corpus: 词 ID 组成的一维列表/数组
        vocab_size: 词表大小
        window_size: 单侧窗口半宽
    返回:
        co_matrix: (vocab_size, vocab_size) 整数矩阵
    """
    co_matrix = np.zeros((vocab_size, vocab_size), dtype=np.int32)
    corpus_len = len(corpus)
    
    for idx, word_id in enumerate(corpus):
        for i in range(1, window_size + 1):
            left_idx = idx - i
            right_idx = idx + i
            if left_idx >= 0:
                co_matrix[word_id, corpus[left_idx]] += 1
            if right_idx < corpus_len:
                co_matrix[word_id, corpus[right_idx]] += 1
                
    return co_matrix

# 测试验证
raw_text = "you say goodbye and i say hello ."
tokens = raw_text.lower().split()
w2id, id2w = build_vocab(tokens)
corpus_ids = [w2id[w] for w in tokens]
C = create_co_matrix(corpus_ids, vocab_size=len(w2id), window_size=1)

print("词表大小:", len(w2id))
print("共现矩阵 C 形状:", C.shape)
print("部分共现计数 ('say' 与 'you'):", C[w2id['say'], w2id['you']])


---
## 模块二：正点互信息 (PPMI) 纯向量化广播手撕

### 【笔试考点与陷阱】
- **数学公式**：
  $$\text{PMI}(x, y) = \log_2 \left( \frac{C(x, y) \cdot N}{S(x) \cdot S(y)} \right), \quad \text{PPMI} = \max(0, \text{PMI})$$
- **踩坑点**：若在 $10000 \times 10000$ 矩阵上使用两层 `for` 循环，笔试运行必超时！必须利用 NumPy **广播机制 (Broadcasting)** 在一行内并行计算外积 $S(x) \cdot S(y)$。


In [ ]:
def ppmi_vectorized(C, eps=1e-8):
    """
    全向量化快速计算 PPMI 矩阵 (无 Python for 循环)
    参数:
        C: 共现矩阵 (V, V)
        eps: 数值平滑常数，防止 log2(0) 产生 -inf 或除以 0
    返回:
        M: 正点互信息矩阵 (V, V)
    """
    N = np.sum(C)                      # 语料库总共现次数 (标量)
    S = np.sum(C, axis=1)              # 每个词的边缘频次向量 (V,)
    
    # 核心矩阵广播外积: S[:, None] 是 (V, 1), S[None, :] 是 (1, V)
    # 相乘得到 (V, V) 的联合独立期望分布矩阵
    expected = (S[:, None] * S[None, :]) / N
    
    # 广播计算 PMI
    pmi = np.log2((C + eps) / (expected + eps))
    
    # 截断负值，得到 PPMI
    ppmi = np.maximum(0, pmi)
    return ppmi.astype(np.float32)

# 测试验证
M = ppmi_vectorized(C)
print("PPMI 矩阵形状:", M.shape)
print("无共现词对 PPMI 严格截断为 0:", M[w2id['you'], w2id['hello']])
print("强共现词对 PPMI 值 > 0:", np.round(M[w2id['say'], w2id['you']], 4))


---
## 模块三：截断 SVD 降维稠密向量与余弦相似度计算

### 【笔试考点与陷阱】
1. **SVD 分解**：$M = U \Sigma V^T$，取左奇异矩阵的前 $k$ 列作为词的低维稠密嵌入；
2. **余弦相似度**：必须加 $\epsilon$ 防止分母模长乘积为 0。


In [ ]:
def cos_similarity(x, y, eps=1e-8):
    """
    计算两个向量的余弦相似度: cos = (x · y) / (||x|| * ||y||)
    """
    nx = x / (np.linalg.norm(x) + eps)
    ny = y / (np.linalg.norm(y) + eps)
    return np.dot(nx, ny)

def svd_embedding(M, dim=2):
    """
    对 PPMI 矩阵执行 SVD 截断降维
    参数:
        M: PPMI 矩阵 (V, V)
        dim: 降维后的稠密维度
    返回:
        word_vecs: 稠密词向量矩阵 (V, dim)
    """
    # np.linalg.svd: U 形状为 (V, V), S 为 (V,), Vh 为 (V, V)
    U, S, Vh = np.linalg.svd(M)
    # 取 U 的前 dim 列作为稠密词向量
    word_vecs = U[:, :dim]
    return word_vecs

# 测试验证
vecs = svd_embedding(M, dim=2)
print("SVD 降维后词向量矩阵形状:", vecs.shape)
sim_you_say = cos_similarity(vecs[w2id['you']], vecs[w2id['say']])
sim_you_goodbye = cos_similarity(vecs[w2id['you']], vecs[w2id['goodbye']])
print(f"cos_sim('you', 'say'): {sim_you_say:.4f}")
print(f"cos_sim('you', 'goodbye'): {sim_you_goodbye:.4f}")


---
## 模块四：Mikolov 高频词二次物理下采样 (Subsampling)

### 【笔试考点与陷阱】
- **保留概率公式**：
  $$P_{\text{keep}}(w) = \sqrt{\frac{t}{f(w)}}, \quad \text{其中 } f(w) = \frac{\text{count}(w)}{\text{total\_words}}$$
- **核心时序认知**：**必须在切滑动窗口前完成物理删除**，这样能直接缩短句子长度，拉近核心实体词之间的上下文跨度。


In [ ]:
def subsample_corpus(words, threshold=1e-4, seed=42):
    """
    高频词物理下采样
    参数:
        words: 文本单词列表
        threshold: 阈值 t，通常 1e-4 或 1e-5
    返回:
        subsampled_words: 下采样后的过滤文本列表
    """
    rng = np.random.RandomState(seed)
    counts = collections.Counter(words)
    total_len = len(words)
    
    subsampled = []
    for w in words:
        freq = counts[w] / total_len
        if freq > threshold:
            p_keep = np.sqrt(threshold / freq)
            if rng.rand() < p_keep:
                subsampled.append(w)
        else:
            subsampled.append(w)
            
    return subsampled

# 模拟超大长文本测试保留率
dummy_words = ["the"] * 1000 + ["apple"] * 20 + ["quantum"] * 5
filtered = subsample_corpus(dummy_words, threshold=1e-3)
print(f"原始词数: {len(dummy_words)}, 下采样后词数: {len(filtered)}")
print(f"超高频词 'the' 保留数: {filtered.count('the')} / 1000 (大量丢弃)")
print(f"低频词 'quantum' 保留数: {filtered.count('quantum')} / 5 (100% 保留)")


---
## 模块五：工业级负采样器 (0.75 次幂平滑 + 100 万槽位轮盘查表法)

### 【笔试考点与陷阱】
1. **0.75 次幂平滑公式**：$P_n(w) = \frac{[\text{count}(w)]^{0.75}}{\sum [\text{count}(w')]^{0.75}}$；
2. **为什么绝不能用 `np.random.choice`？**：它每次都要做一次累积分布函数二分查找，复杂度 $O(V)$。
3. **100 万槽位轮盘表 (Unigram Table)**：预先分配大小为 $1,000,000$ 的整数数组，训练时一步整数随机索引查表，复杂度 $O(1)$！


In [ ]:
class FastUnigramSampler:
    def __init__(self, corpus_ids, power=0.75, table_size=1000000):
        self.table_size = table_size
        counts = collections.Counter(corpus_ids)
        vocab_size = len(counts)
        
        # 1. 计算 0.75 次幂平滑分布
        count_arr = np.zeros(vocab_size, dtype=np.float64)
        for wid, cnt in counts.items():
            count_arr[wid] = cnt
        p = np.power(count_arr, power)
        p = p / np.sum(p)
        
        # 2. 预建 100 万槽位离线查找表 (Table Lookup)
        self.table = np.zeros(table_size, dtype=np.int32)
        idx = 0
        for wid, prob in enumerate(p):
            slots = int(round(prob * table_size))
            if idx + slots > table_size:
                slots = table_size - idx
            self.table[idx : idx + slots] = wid
            idx += slots
            
    def get_negative_samples(self, batch_size, K):
        """O(1) 极速动态负采样"""
        rand_idx = np.random.randint(0, self.table_size, size=(batch_size, K))
        return self.table[rand_idx]

# 测试验证采样器
sampler = FastUnigramSampler(corpus_ids, power=0.75, table_size=100000)
neg_samples = sampler.get_negative_samples(batch_size=4, K=3)
print("动态采样的负样本矩阵 (Batch=4, K=3):\n", neg_samples)


---
## 模块六：Embedding 核心层手撕与【梯度覆盖毁灭陷阱】

### 【笔试考点与高危雷区】
> [!CAUTION]
> **雷区警告**：在反向传播更新权重矩阵 $W$ 时，**同一个 Batch 中很可能包含重复出现的词 ID**！
> - 若直接写 `dW[idx] = dout`，后出现的词梯度会直接把前面出现的词梯度**强行覆盖清空**！
> - **正解**：必须使用原子累加函数 `np.add.at(self.dW, self.idx, dout)`！


In [ ]:
class Embedding:
    def __init__(self, W):
        self.W = W
        self.dW = np.zeros_like(W)
        self.idx = None
        
    def forward(self, idx):
        self.idx = idx
        return self.W[idx]
        
    def backward(self, dout):
        self.dW[...] = 0
        # 核心考点: 必须使用原子级累加 np.add.at 防止重复词 ID 覆盖梯度!
        np.add.at(self.dW, self.idx, dout)
        return None

# 测试梯度覆盖防范机制
W_dummy = np.zeros((5, 2), dtype=np.float32)
embed = Embedding(W_dummy)

# 模拟同一个 batch 中连续出现 3 次 ID=1 的单词
repeat_indices = np.array([1, 1, 1])
douts = np.array([[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]], dtype=np.float32)

embed.forward(repeat_indices)
embed.backward(douts)

print("ID=1 的累加梯度为:", embed.dW[1])
assert np.allclose(embed.dW[1], [6.0, 6.0]), "错误: 梯度被覆盖了!"
print(">>> 测试通过: 原子累加正确，无梯度覆盖！")


---
## 模块七：CBOW + 负采样端到端核心实现 (CBOW with NEG)

### 【笔试考点与公式推导】
1. **隐层聚合**：$h = \frac{1}{2c} \sum v_i$；
2. **正样本二分类损失**：$- \log \sigma(u_{\text{pos}}^T h)$，导数：$(y_{\text{pos}} - 1) h$；
3. **负样本二分类损失**：$- \sum \log \sigma(-u_{\text{neg}}^T h)$，导数：$y_{\text{neg}} h$；
4. **回传隐层总梯度**：$\frac{\partial L}{\partial h} = (y_{\text{pos}} - 1) u_{\text{pos}} + \sum y_{\text{neg}} u_{\text{neg}}$。


In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20.0, 20.0)))

class SimpleCBOW_NEG:
    def __init__(self, vocab_size, hidden_dim):
        self.W_in = np.random.randn(vocab_size, hidden_dim) * 0.05
        self.W_out = np.random.randn(vocab_size, hidden_dim) * 0.05
        
    def forward(self, contexts, targets, neg_samples):
        """
        contexts: (B, 2c)
        targets: (B,)
        neg_samples: (B, K)
        """
        self.B, self.C_len = contexts.shape
        self.K = neg_samples.shape[1]
        self.contexts = contexts
        self.targets = targets
        self.neg_samples = neg_samples
        
        # 1. 隐层平均池化
        v_ctx = self.W_in[contexts]                    # (B, 2c, H)
        self.h = np.mean(v_ctx, axis=1)                 # (B, H)
        
        # 2. 正样本前向 (标签 t = 1)
        u_pos = self.W_out[targets]                     # (B, H)
        score_pos = np.sum(self.h * u_pos, axis=1)      # (B,)
        self.y_pos = sigmoid(score_pos)                 # (B,)
        loss_pos = -np.sum(np.log(self.y_pos + 1e-7))
        
        # 3. 负样本前向 (标签 t = 0)
        u_neg = self.W_out[neg_samples]                 # (B, K, H)
        score_neg = np.sum(self.h[:, None, :] * u_neg, axis=2) # (B, K)
        self.y_neg = sigmoid(score_neg)                 # (B, K)
        loss_neg = -np.sum(np.log(1.0 - self.y_neg + 1e-7))
        
        return (loss_pos + loss_neg) / self.B

    def backward(self, lr=0.01):
        # 1. 正负样本梯度标量误差 (y - t)
        dy_pos = (self.y_pos - 1.0) / self.B            # (B,)
        dy_neg = self.y_neg / self.B                    # (B, K)
        
        # 2. 对 W_out[targets] 与 W_out[neg_samples] 的梯度
        grad_u_pos = dy_pos[:, None] * self.h           # (B, H)
        grad_u_neg = dy_neg[:, :, None] * self.h[:, None, :] # (B, K, H)
        
        # 3. 对隐层 h 的回传梯度
        u_pos = self.W_out[self.targets]                # (B, H)
        u_neg = self.W_out[self.neg_samples]            # (B, K, H)
        dh = dy_pos[:, None] * u_pos + np.sum(dy_neg[:, :, None] * u_neg, axis=1) # (B, H)
        
        # 4. 均摊反向传播给每一个上下文词
        dv_ctx = dh / self.C_len                        # (B, H)
        
        # 5. 原地更新参数 (防覆盖原子累加)
        np.add.at(self.W_out, self.targets, -lr * grad_u_pos)
        for k in range(self.K):
            np.add.at(self.W_out, self.neg_samples[:, k], -lr * grad_u_neg[:, k])
            
        for c in range(self.C_len):
            np.add.at(self.W_in, self.contexts[:, c], -lr * dv_ctx)

# 快速验证运行
cbow = SimpleCBOW_NEG(vocab_size=10, hidden_dim=4)
dummy_contexts = np.array([[0, 2], [1, 3]])
dummy_targets = np.array([1, 2])
dummy_negs = np.array([[4, 5], [6, 7]])

loss = cbow.forward(dummy_contexts, dummy_targets, dummy_negs)
print("CBOW 前向单步 Loss:", round(loss, 4))
cbow.backward(lr=0.05)
print(">>> CBOW 反向传播成功，参数已更新！")


---
## 模块八：Skip-Gram + 负采样端到端核心实现 (Skip-Gram with NEG)

### 【笔试考点与差异】
- **无池化单挑机制**：输入隐层直接为中心词向量 $h = v_c$，不进行 $\frac{1}{2c}$ 的平均稀释，梯度全量直接回传给中心词。


In [ ]:
class SimpleSkipGram_NEG:
    def __init__(self, vocab_size, hidden_dim):
        self.W_in = np.random.randn(vocab_size, hidden_dim) * 0.05
        self.W_out = np.random.randn(vocab_size, hidden_dim) * 0.05
        
    def forward(self, centers, targets, neg_samples):
        self.B = centers.shape[0]
        self.centers = centers
        self.targets = targets
        self.neg_samples = neg_samples
        self.K = neg_samples.shape[1]
        
        # 1. 隐层直接取中心词，无平均池化！
        self.v_center = self.W_in[centers]               # (B, H)
        
        # 2. 正负样本得分与二分类预测
        u_pos = self.W_out[targets]                     # (B, H)
        self.y_pos = sigmoid(np.sum(self.v_center * u_pos, axis=1)) # (B,)
        
        u_neg = self.W_out[neg_samples]                 # (B, K, H)
        self.y_neg = sigmoid(np.sum(self.v_center[:, None, :] * u_neg, axis=2)) # (B, K)
        
        loss = -np.sum(np.log(self.y_pos + 1e-7)) - np.sum(np.log(1.0 - self.y_neg + 1e-7))
        return loss / self.B

    def backward(self, lr=0.01):
        dy_pos = (self.y_pos - 1.0) / self.B            # (B,)
        dy_neg = self.y_neg / self.B                    # (B, K)
        
        grad_u_pos = dy_pos[:, None] * self.v_center    # (B, H)
        grad_u_neg = dy_neg[:, :, None] * self.v_center[:, None, :] # (B, K, H)
        
        u_pos = self.W_out[self.targets]
        u_neg = self.W_out[self.neg_samples]
        # 中心词独自接收所有梯度
        grad_v_center = dy_pos[:, None] * u_pos + np.sum(dy_neg[:, :, None] * u_neg, axis=1)
        
        np.add.at(self.W_out, self.targets, -lr * grad_u_pos)
        for k in range(self.K):
            np.add.at(self.W_out, self.neg_samples[:, k], -lr * grad_u_neg[:, k])
        np.add.at(self.W_in, self.centers, -lr * grad_v_center)

# 快速验证运行
sg = SimpleSkipGram_NEG(vocab_size=10, hidden_dim=4)
dummy_centers = np.array([1, 2])
loss = sg.forward(dummy_centers, dummy_targets, dummy_negs)
print("Skip-Gram 前向单步 Loss:", round(loss, 4))
sg.backward(lr=0.05)
print(">>> Skip-Gram 反向传播成功，参数已更新！")


---
## 模块九：GloVe 全局加权最小二乘回归手撕 (GloVe Weighted Least Squares)

### 【笔试考点】
1. **加权函数**：$f(X_{ij}) = \min(1, (X_{ij} / x_{\max})^\alpha)$，通常取 $x_{\max}=100, \alpha=0.75$；
2. **损失函数**：$J = \sum f(X_{ij}) (w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij})^2$；
3. **稀疏性优化**：只对共现矩阵中的非零项（Non-zero elements）遍历更新。


In [ ]:
class SimpleGloVe:
    def __init__(self, vocab_size, hidden_dim, x_max=100.0, alpha=0.75):
        self.W = np.random.randn(vocab_size, hidden_dim) * 0.05
        self.W_tilde = np.random.randn(vocab_size, hidden_dim) * 0.05
        self.b = np.zeros(vocab_size)
        self.b_tilde = np.zeros(vocab_size)
        self.x_max = x_max
        self.alpha = alpha
        
    def train_step(self, i, j, X_ij, lr=0.01):
        """对单个非零共现对执行加权最小二乘回归更新"""
        # 1. 计算加权权重 f(X_ij)
        weight = (X_ij / self.x_max) ** self.alpha if X_ij < self.x_max else 1.0
        
        # 2. 前向预测与误差
        pred = np.dot(self.W[i], self.W_tilde[j]) + self.b[i] + self.b_tilde[j]
        diff = pred - np.log(X_ij)
        loss = weight * (diff ** 2)
        
        # 3. 梯度推导
        grad_common = 2.0 * weight * diff
        grad_W = grad_common * self.W_tilde[j]
        grad_W_tilde = grad_common * self.W[i]
        grad_b = grad_common
        grad_b_tilde = grad_common
        
        # 4. 更新参数
        self.W[i] -= lr * grad_W
        self.W_tilde[j] -= lr * grad_W_tilde
        self.b[i] -= lr * grad_b
        self.b_tilde[j] -= lr * grad_b_tilde
        
        return loss

# 快速验证运行
glove = SimpleGloVe(vocab_size=5, hidden_dim=2)
loss = glove.train_step(i=0, j=1, X_ij=15.0, lr=0.01)
print(f"GloVe 单步加权回归 Loss: {loss:.4f}")
print(">>> GloVe 更新成功！")


---
## 模块十：现代动态上下文表征——缩放点积自注意力 (Self-Attention)

### 【笔试考点与降维打击】
- **静态词向量致命弱点**：一词多义导致流形坍缩；
- **自注意力公式**：
  $$\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{QK^T}{\sqrt{d_k}} \right) V$$
- **消歧机制**：同一个词通过与全句所有词的点积交互，动态输出专属于当前语境的高阶表征！


In [ ]:
def softmax(x, axis=-1):
    ex = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return ex / np.sum(ex, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """
    缩放点积自注意力机制
    输入形状: (Seq_Len, Dim)
    输出形状: (Seq_Len, Dim), 注意力矩阵 (Seq_Len, Seq_Len)
    """
    d_k = Q.shape[-1]
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    context_vectors = np.dot(weights, V)
    return context_vectors, weights

# 模拟一词多义消歧: "apple" 出现在两个不同句子中
dim = 4
# 构造静态表征: apple(同坐标), fruit(水果), store(商店)
v_apple = np.array([0.5, 0.5, 0.0, 0.0])
v_fruit = np.array([0.9, 0.8, 0.0, 0.0])
v_store = np.array([0.0, 0.0, 0.8, 0.9])

# 句子 A: [fruit, apple]
seq_A = np.stack([v_fruit, v_apple])
dyn_A, w_A = scaled_dot_product_attention(seq_A, seq_A, seq_A)

# 句子 B: [store, apple]
seq_B = np.stack([v_store, v_apple])
dyn_B, w_B = scaled_dot_product_attention(seq_B, seq_B, seq_B)

print("静态初始时: apple 仅有 1 个固定向量:", v_apple)
print("句子 A 经自注意力重构后的 apple 动态向量:", np.round(dyn_A[1], 3))
print("句子 B 经自注意力重构后的 apple 动态向量:", np.round(dyn_B[1], 3))
print(">>> 成功演示动态上下文表征消歧效果！")


---
## 企业笔试手撕核心口诀与雷区速记卡

```
1. PPMI 广播加速: (S[:, None] * S[None, :]) / N，坚决不用两层 for 循环！
2. 负采样 0.75 次幂: 凸幂平滑压高频提低频，预建 100 万槽位 table 实现 O(1) 随机抽取。
3. 反向传播防覆盖: np.add.at(dW, idx, dout) 是唯一真理，直接赋值 dW[idx]=dout 必得零分！
4. CBOW vs Skip-Gram: CBOW 隐层平均被高频词带偏，Skip-Gram 点对点单挑保留罕见词专属坐标。
5. 静态 vs 动态: Word2Vec/GloVe 是查表静态投影，Transformer 自注意力动态编织消除一词多义。
```

